In [3]:
from pathlib import Path
import pandas as pd

cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_output = project_root / "data" / "procesed"
parquet_path = path_output / "master_2019_1M_per_month.parquet"

df = pd.read_parquet(parquet_path)


In [4]:
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000000 entries, 0 to 11999999
Data columns (total 21 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   DOLocationID           int64  
 1   PULocationID           int64  
 2   RatecodeID             float64
 3   VendorID               float64
 4   congestion_surcharge   float64
 5   extra                  float64
 6   fare_amount            float64
 7   improvement_surcharge  float64
 8   mta_tax                float64
 9   passenger_count        float64
 10  payment_type           float64
 11  store_and_fwd_flag     object 
 12  tip_amount             float64
 13  tolls_amount           float64
 14  total_amount           float64
 15  tpep_dropoff_datetime  object 
 16  tpep_pickup_datetime   object 
 17  trip_distance          float64
 18  year                   int64  
 19  month                  int64  
 20  source_file            object 
dtypes: float64(13), int64(4), object(4)
memory usage: 1.

((12000000, 21), None)

In [5]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(25)

congestion_surcharge     0.052736
VendorID                 0.003084
RatecodeID               0.003084
store_and_fwd_flag       0.003084
payment_type             0.003084
passenger_count          0.003084
DOLocationID             0.000000
PULocationID             0.000000
extra                    0.000000
mta_tax                  0.000000
improvement_surcharge    0.000000
fare_amount              0.000000
tip_amount               0.000000
tolls_amount             0.000000
total_amount             0.000000
tpep_dropoff_datetime    0.000000
tpep_pickup_datetime     0.000000
trip_distance            0.000000
year                     0.000000
month                    0.000000
source_file              0.000000
dtype: float64

extraño que todos los unicas variables con perdidos, compartan el mismo porcentaje, probablemente sea error del sampleo pero solo con 0.004 de mi muestreo total no representan ni un 5 porciento podriamos simplemente eliminarlos pues son valores de diferentes variables que simplemente esatn contaminados. podemos eliminarlos ? 

testeo, para ver si son las mismas rows en esas columnas, protegeremos congestion subcharge

In [6]:
core_vendor_cols = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
    "passenger_count"
]
mask_block_missing = df[core_vendor_cols].isna().all(axis=1)

print("Rows fully missing vendor block:",
      mask_block_missing.sum(),
      "| %:", mask_block_missing.mean())

Rows fully missing vendor block: 37009 | %: 0.003084083333333333


son exactamente las misma columans, puede ser un error de ingesta o problemas de sistema, decidimos eliminarlos 

In [7]:
df = df.loc[~mask_block_missing].copy()

lets work the time variables 

In [ ]:
for c in ["tpep_pickup_datetime","tpep_dropoff_datetime"]:
    if df[c].dtype == "object":
        df[c] = pd.to_datetime(df[c], errors="coerce")
        

In [ ]:
df = df.dropna(subset=["tpep_pickup_datetime","tpep_dropoff_datetime"])
df = df.drop(columns=["year"])

we can start with feature ingeniering to help model generalize better 

In [10]:
df["duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)

we keep only trip that are physically real 

In [11]:
physics_mask = (
    (df["duration_min"] > 0) &
    (df["trip_distance"] > 0) &
    (df["total_amount"] > 0)
)

df = df.loc[physics_mask].copy()

Now handle extreme physics:
NYC upper realistic bounds:
Speed > 120 mph → impossible
Duration > 6 hours → extremely unlikely taxi ride

In [12]:
df = df[
    (df["speed_mph"] > 0) &
    (df["speed_mph"] < 120) &
    (df["duration_min"] < 6*60)
].copy()

In [13]:
df.isna().mean().sort_values(ascending=False).head(15)

congestion_surcharge     0.053052
PULocationID             0.000000
DOLocationID             0.000000
RatecodeID               0.000000
VendorID                 0.000000
extra                    0.000000
fare_amount              0.000000
improvement_surcharge    0.000000
mta_tax                  0.000000
passenger_count          0.000000
payment_type             0.000000
store_and_fwd_flag       0.000000
tip_amount               0.000000
tolls_amount             0.000000
total_amount             0.000000
dtype: float64

perfect, we just have with missing values the congestion that we will make it a true and false to drop the original one.

is a weird case bc had almost 10 mill values at 2.4-2.6 range, almost 1 mill missings and the rest 0-0.2, so we conclude Empirically you observed:

~10M values clustered around 2.5
The rest clustered around 0
This is not continuous variability.

This is:

small rounding noise
X∈{0,2.5}+small rounding noise

So mathematically this variable behaves like a Bernoulli random variable disguised as numeric.

we will create a boolean based on if it pays or not and based on it distrubution we chose these condition  

In [15]:
df["has_congestion_fee"] = (df["congestion_surcharge"] > 1).astype(int)
df = df.drop(columns=["congestion_surcharge"])

more feature ing 

In [20]:
import numpy as np
df["fare_per_mile"] = df["fare_amount"] / df["trip_distance"]
df["total_per_mile"] = df["total_amount"] / df["trip_distance"]
df["tip_pct"] = df["tip_amount"] / df["fare_amount"].replace(0, np.nan)
df["toll_ratio"] = df["tolls_amount"] / df["total_amount"].replace(0, np.nan)

time 


In [21]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["weekday"] = df["tpep_pickup_datetime"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

C yclical encoding (critical for rigor)

In [22]:
df["hour_sin"] = np.sin(2*np.pi*df["pickup_hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["pickup_hour"]/24)

df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

spatial Structure

In [39]:
path_output_zones = project_root / "data" / "raw" / "taxi_zone_lookup.csv"
zones = pd.read_csv(path_output_zones)

zones = zones.rename(columns={"LocationID":"PULocationID"})
df = df.merge(zones[["PULocationID","Borough"]], on="PULocationID", how="left")
df = df.rename(columns={"Borough":"PU_Borough"})

zones = zones.rename(columns={"PULocationID":"DOLocationID"})
df = df.merge(zones[["DOLocationID","Borough"]], on="DOLocationID", how="left")
df = df.rename(columns={"Borough":"DO_Borough"})

Trip Efficiency Features

In [30]:
df["log_duration"] = np.log1p(df["duration_min"])
df["log_distance"] = np.log1p(df["trip_distance"])

Behavioral Deviation Features

In [40]:
baseline = df.groupby(["pickup_hour","PU_Borough"])["speed_mph"].median()
df["expected_speed"] = df.set_index(["pickup_hour","PU_Borough"]).index.map(baseline)

df["speed_deviation"] = df["speed_mph"] - df["expected_speed"]